# Ekstraksi dan Menampilkan Matriks Kernel

Notebook ini dibuat untuk memuat file `kernel_history.pkl` (atau `kernel_history_all.pkl`) dan menampilkan seluruh nilai matriks untuk setiap filter pada setiap channel di semua epoch.

In [1]:
import pickle
import numpy as np
import torch
import os

# Deteksi file history kernel
file_path = 'kernel_history.pkl'
if not os.path.exists(file_path):
    file_path = 'kernel_history_all.pkl'

print(f"Memuat data dari: {file_path} ...")
with open(file_path, 'rb') as f:
    kernel_history_all = pickle.load(f)

print("Berhasil memuat data!")
print(f"Jumlah layer yang terekam: {len(kernel_history_all)}")

Memuat data dari: kernel_history.pkl ...
Berhasil memuat data!
Jumlah layer yang terekam: 39


### Menampilkan Seluruh Matriks

**PERINGATAN**: Menampilkan seluruh matriks secara langsung akan menghasilkan output yang sangat panjang (jutaan baris teks karena total datanya sangat besar). Kode di bawah ini akan melakukan iterasi ke seluruh struktur secara hierarkis:
`Layer -> Epoch -> Filter -> Channel -> Nilai Matriks`

In [2]:
import sys
from io import StringIO

np.set_printoptions(threshold=np.inf, linewidth=200, suppress=True)
torch.set_printoptions(profile="full", linewidth=200, sci_mode=False)


def print_all_kernels(history_dict, max_layers=None, max_epochs=None):
    """
    Optimized: buffers all output into StringIO and flushes once per layer,
    minimizing I/O calls and redundant tensor conversions.
    """
    for layer_idx, (layer_name, epochs_data) in enumerate(history_dict.items()):
        if max_layers is not None and layer_idx >= max_layers:
            break

        buf = StringIO()  # ✅ Buffer output per layer — one write at the end
        buf.write("=" * 80 + "\n")
        buf.write(f"LAYER: {layer_name}\n")
        buf.write("=" * 80 + "\n")

        for epoch_idx, epoch_tensor in enumerate(epochs_data):
            if max_epochs is not None and epoch_idx >= max_epochs:
                break

            buf.write(f"\n  --- EPOCH {epoch_idx + 1} ---\n")

            # ✅ Convert the ENTIRE tensor to numpy ONCE per epoch
            epoch_np = epoch_tensor.detach().cpu().numpy()

            ndim = epoch_np.ndim
            num_filters = epoch_np.shape[0]
            num_channels = epoch_np.shape[1] if ndim > 1 else 1

            for f_idx in range(num_filters):
                buf.write(f"\n    [Filter {f_idx}]\n")
                for c_idx in range(num_channels):
                    buf.write(f"      > Channel {c_idx}:\n")

                    # ✅ Slice numpy array directly — no repeated .numpy() calls
                    if ndim == 4:
                        matrix = epoch_np[f_idx, c_idx]
                    elif ndim == 3:
                        matrix = epoch_np[f_idx, c_idx]
                    else:
                        matrix = epoch_np[f_idx]

                    # ✅ Format once, indent once
                    matrix_str = np.array2string(matrix, separator=", ", prefix=" " * 8)
                    buf.write(f"        {matrix_str}\n")

            buf.write("\n" + "-" * 50 + "\n")

        # ✅ Single sys.stdout.write per layer instead of thousands of print()
        sys.stdout.write(buf.getvalue())
        sys.stdout.flush()


print("Mencetak seluruh nilai matriks...")
print_all_kernels(kernel_history_all, max_layers=1, max_epochs=1)

Mencetak seluruh nilai matriks...
LAYER: model.0.conv

  --- EPOCH 1 ---

    [Filter 0]
      > Channel 0:
        [[-0.06393809,  0.08260848, -0.06781896],
         [ 0.18536964,  0.05558015, -0.16967435],
         [ 0.02601405, -0.32131463,  0.33239606]]
      > Channel 1:
        [[-0.03464707,  0.23073933, -0.09405521],
         [ 0.7611679 , -0.39586765, -0.5148893 ],
         [ 0.02534052, -1.292255  ,  1.2591463 ]]
      > Channel 2:
        [[-0.01175002,  0.058996  , -0.0218409 ],
         [ 0.15362445, -0.04332642, -0.16124968],
         [ 0.01057352, -0.25686237,  0.2950396 ]]

    [Filter 1]
      > Channel 0:
        [[ 0.014218  ,  0.32438773,  0.29462773],
         [-0.11407483, -0.7486328 , -0.6360403 ],
         [ 0.07636148,  0.451051  ,  0.31975272]]
      > Channel 1:
        [[ 0.1769964 ,  1.514567  ,  1.088578  ],
         [-0.24052635, -2.7168868 , -1.6848683 ],
         [ 0.04527291,  1.191277  ,  0.63689655]]
      > Channel 2:
        [[ 0.02342012,  0.26042

### Opsi Tambahan: Menyimpan ke File Teks
Jika output di Jupyter Notebook lag/crash karena terlalu panjang, Anda bisa menjalankan sel di bawah ini untuk mengekspor (save) seluruh nilai tersebut ke file `TXT` per layer.

In [3]:
import os
import numpy as np
from multiprocessing import Pool  # ✅ True parallelism, bukan thread
from io import StringIO


def _write_layer_worker(args):
    layer_name, epochs_list, export_dir = args

    safe_name = layer_name.replace(".", "_")
    file_path = os.path.join(export_dir, f"{safe_name}.txt")

    buf = StringIO()
    buf.write("=" * 50 + "\n")
    buf.write(f"LAYER: {layer_name}\n")
    buf.write("=" * 50 + "\n")

    for epoch_idx, epoch_np in enumerate(epochs_list):
        buf.write(f"\n--- EPOCH {epoch_idx + 1} ---\n")
        ndim = epoch_np.ndim
        num_filters = epoch_np.shape[0]
        num_channels = epoch_np.shape[1] if ndim > 1 else 1

        for f_idx in range(num_filters):
            buf.write(f"\n  [Filter {f_idx}]\n")
            for c_idx in range(num_channels):
                buf.write(f"    > Channel {c_idx}:\n")
                if ndim >= 3:
                    matrix = epoch_np[f_idx, c_idx]
                else:
                    matrix = epoch_np[f_idx]
                matrix_str = np.array2string(matrix, separator=", ", prefix=" " * 6)
                buf.write(f"      {matrix_str}\n")

    # Single write
    with open(file_path, "w", encoding="utf-8", buffering=1024 * 1024) as f:
        f.write(buf.getvalue())

    return layer_name


def export_kernels_to_txt_fast(history_dict, export_dir="kernels"):
    os.makedirs(export_dir, exist_ok=True)

    # ✅ Convert tensor → numpy SEBELUM masuk multiprocessing
    # (tensor PyTorch tidak bisa di-pickle langsung)
    tasks = []
    for layer_name, epochs_data in history_dict.items():
        epochs_np = [t.detach().cpu().numpy() for t in epochs_data]
        tasks.append((layer_name, epochs_np, export_dir))

    print(f"Mengekspor {len(tasks)} layer dengan {os.cpu_count()} proses...")

    with Pool(processes=os.cpu_count()) as pool:
        for layer_name in pool.imap_unordered(_write_layer_worker, tasks):
            print(f"  ✓ {layer_name}")

    print(f"\nSelesai! Export ke: {export_dir}")


export_kernels_to_txt_fast(kernel_history_all)

Mengekspor 39 layer dengan 16 proses...
  ✓ model.0.conv
  ✓ model.2.m.0.cv2.conv
  ✓ model.2.m.0.cv1.conv
  ✓ model.1.conv
  ✓ model.4.m.0.cv2.conv
  ✓ model.2.cv1.conv
  ✓ model.4.m.0.cv1.conv
  ✓ model.6.m.0.m.0.cv1.conv
  ✓ model.6.m.0.cv2.conv
  ✓ model.6.m.0.cv1.conv
  ✓ model.6.m.0.m.0.cv2.conv
  ✓ model.6.m.0.m.1.cv1.conv
  ✓ model.2.cv2.conv
  ✓ model.6.m.0.m.1.cv2.conv
  ✓ model.6.m.0.cv3.conv
  ✓ model.4.cv1.conv
  ✓ model.3.conv
  ✓ model.8.m.0.cv1.conv
  ✓ model.8.m.0.m.0.cv2.conv
  ✓ model.8.m.0.cv2.conv
  ✓ model.9.m.0.attn.pe.conv
  ✓ model.8.m.0.m.0.cv1.conv
  ✓ model.8.m.0.m.1.cv1.conv
  ✓ model.8.m.0.m.1.cv2.conv
  ✓ model.4.cv2.conv
  ✓ model.6.cv1.conv
  ✓ model.8.m.0.cv3.conv
  ✓ model.6.cv2.conv
  ✓ model.9.m.0.attn.proj.conv
  ✓ model.5.conv
  ✓ model.9.m.0.ffn.0.conv
  ✓ model.9.m.0.attn.qkv.conv
  ✓ model.9.m.0.ffn.1.conv
  ✓ model.7.conv
  ✓ model.8.cv1.conv
  ✓ model.9.cv1.conv
  ✓ model.9.cv2.conv
  ✓ model.8.cv2.conv
  ✓ model.10.conv.conv

Selesai! Export